In [2]:
import pandas as pd
import numpy as np
import os
import re

# -------------------------------------------------------------------
# Load base file
# -------------------------------------------------------------------
base_path = r"C:\Users\HELIOS-300\Desktop\WAVES\AM Full Code\Cameron_AM_Clean.csv"
base_df = pd.read_csv(base_path, low_memory=False)
print(f"Base file loaded: {base_df.shape}")
print("Columns:", base_df.columns.tolist())
print("Unique IDs:", sorted(base_df["id"].unique()))


Base file loaded: (356191, 13)
Columns: ['id', 'obs', 'date', 'date_time', 'rel_time', 'activity_type', 'broad_domain', 'broad.behavior_do', 'posture_wbm', 'posture_broad', 'broad.posture_do', 'sed.posture_do', 'intensity_do']
Unique IDs: ['AM01', 'AM02', 'AM03', 'AM04', 'AM05', 'AM06', 'AM07', 'AM08', 'AM09', 'AM10', 'AM11', 'AM12', 'AM13', 'AM14', 'AM15', 'AM16', 'AM17', 'AM18', 'AM19', 'AM20', 'AM21', 'AM22', 'AM24', 'AM25', 'AM26', 'AM27']


In [3]:
# -------------------------------------------------------------------
# Load and concat all activPal AM files whose ID exists in the base file
# Naming pattern: AP_AM01-...
# -------------------------------------------------------------------
activpal_folder = r"C:\Users\HELIOS-300\Desktop\Data\activPal AM"
valid_ids = set(base_df["id"].unique())

activpal_chunks = []

for fname in sorted(os.listdir(activpal_folder)):
    m = re.match(r"AP_(AM\d+)", fname, re.IGNORECASE)
    if not m:
        continue
    file_id = m.group(1).upper()
    if file_id not in valid_ids:
        print(f"  SKIPPED (not in base): {file_id} | {fname}")
        continue

    path = os.path.join(activpal_folder, fname)
    ap = pd.read_csv(path, sep=";", skiprows=1, low_memory=False)

    ap = ap.drop(columns=["Time"], errors="ignore")
    ap = ap.rename(columns={"Time(approx)": "date_time"})
    ap["id"] = file_id

    activpal_chunks.append(ap)
    print(f"  Loaded {file_id:6s} | {len(ap):>8,} rows")

activpal_df = pd.concat(activpal_chunks, ignore_index=True)
print(f"\nAll activPal combined: {activpal_df.shape}")
print("activPal columns:", activpal_df.columns.tolist())


  Loaded AM01   |  604,813 rows
  Loaded AM02   |  602,247 rows
  Loaded AM03   |  601,545 rows
  Loaded AM04   |  604,816 rows
  Loaded AM05   |   81,503 rows
  Loaded AM06   |  604,823 rows
  Loaded AM07   |  604,818 rows
  Loaded AM08   |  604,820 rows
  Loaded AM09   |  588,299 rows
  Loaded AM10   |  593,669 rows
  Loaded AM11   |  604,819 rows
  Loaded AM12   |  604,816 rows
  Loaded AM13   |  604,958 rows
  Loaded AM14   |  604,818 rows
  Loaded AM15   |  604,816 rows
  Loaded AM16   |  604,813 rows
  Loaded AM17   |  604,815 rows
  Loaded AM18   |  604,820 rows
  Loaded AM19   |  608,163 rows
  Loaded AM20   |  604,816 rows
  Loaded AM21   |  602,321 rows
  Loaded AM22   |  604,819 rows
  SKIPPED (not in base): AM23 | AP_AM23-AP670170 29Jan18 10-00am for 3d 4h 12m-CREA-PB09010394-Epochs1s.csv
  Loaded AM24   |  604,818 rows
  Loaded AM25   |  604,822 rows

All activPal combined: (13959787, 18)
activPal columns: ['date_time', 'StepCount', 'Activity Score (MET.s)', 'Sedentary Tim

In [4]:
# -------------------------------------------------------------------
# Left merge on id + date_time
# -------------------------------------------------------------------
merged_df = base_df.merge(activpal_df, on=["id", "date_time"], how="left")

print(f"Merged shape: {merged_df.shape}")
print(f"Base rows:    {len(base_df):,}  (should be unchanged)")
print()

new_cols = [c for c in activpal_df.columns if c not in ("id", "date_time")]
filled = merged_df[new_cols[0]].notna().sum()
print(f"Rows with activPal data matched: {filled:,} / {len(merged_df):,} ({filled/len(merged_df)*100:.1f}%)")
print()
print("New activPal columns added:", new_cols)


Merged shape: (356191, 29)
Base rows:    356,191  (should be unchanged)

Rows with activPal data matched: 327,972 / 356,191 (92.1%)

New activPal columns added: ['StepCount', 'Activity Score (MET.s)', 'Sedentary Time (s)', 'Upright Time (s)', 'Stepping Time (s)', 'Cycling Time (s)', 'Primary Lying Time (s)', 'Secondary Lying Time (s)', 'Nonwear Time (s)', 'Seated Transport Time (s)', 'Data Errors (s)', 'Sedentary to Upright Movements', 'Upright to Sedentary Movements', 'Sum(abs(dChannel1))', 'Sum(abs(dChannel2))', 'Sum(abs(dChannel3))']


In [5]:
# -------------------------------------------------------------------
# Nullify all activPal columns where intensity_do == "non_codable"
# -------------------------------------------------------------------
activpal_cols = [c for c in merged_df.columns if c not in base_df.columns]

non_codable_mask = merged_df["intensity_do"] == "non_codable"
merged_df.loc[non_codable_mask, activpal_cols] = np.nan

print(f"Rows where intensity_do == non_codable: {non_codable_mask.sum():,}")
print(f"activPal columns nullified for those rows: {activpal_cols}")


Rows where intensity_do == non_codable: 0
activPal columns nullified for those rows: ['StepCount', 'Activity Score (MET.s)', 'Sedentary Time (s)', 'Upright Time (s)', 'Stepping Time (s)', 'Cycling Time (s)', 'Primary Lying Time (s)', 'Secondary Lying Time (s)', 'Nonwear Time (s)', 'Seated Transport Time (s)', 'Data Errors (s)', 'Sedentary to Upright Movements', 'Upright to Sedentary Movements', 'Sum(abs(dChannel1))', 'Sum(abs(dChannel2))', 'Sum(abs(dChannel3))']


In [6]:
# -------------------------------------------------------------------
# Export merged file
# -------------------------------------------------------------------
output_path = r"C:\Users\HELIOS-300\Desktop\WAVES\AM Full Code\am_testing.csv"
merged_df.to_csv(output_path, index=False)
print(f"Exported {len(merged_df):,} rows to {output_path}")
print("Final columns:", merged_df.columns.tolist())


Exported 356,191 rows to C:\Users\HELIOS-300\Desktop\WAVES\AM Full Code\am_testing.csv
Final columns: ['id', 'obs', 'date', 'date_time', 'rel_time', 'activity_type', 'broad_domain', 'broad.behavior_do', 'posture_wbm', 'posture_broad', 'broad.posture_do', 'sed.posture_do', 'intensity_do', 'StepCount', 'Activity Score (MET.s)', 'Sedentary Time (s)', 'Upright Time (s)', 'Stepping Time (s)', 'Cycling Time (s)', 'Primary Lying Time (s)', 'Secondary Lying Time (s)', 'Nonwear Time (s)', 'Seated Transport Time (s)', 'Data Errors (s)', 'Sedentary to Upright Movements', 'Upright to Sedentary Movements', 'Sum(abs(dChannel1))', 'Sum(abs(dChannel2))', 'Sum(abs(dChannel3))']


In [7]:
# -------------------------------------------------------------------
# Summary export: sedentary time + steps by id + obs (GT vs activPal)
# Note: AM has no ground-truth step column — gt_sedentary_s only
# -------------------------------------------------------------------
def summarise_group(g):
    posture_col = g["broad.posture_do"]
    sed_gt = (
        (posture_col == "sedentary").sum()
        if posture_col.notna().any()
        else pd.NA
    )

    step_ap_raw = pd.to_numeric(g["StepCount"], errors="coerce")
    step_ap = step_ap_raw.sum(min_count=1)

    sed_ap_raw = pd.to_numeric(g["Sedentary Time (s)"], errors="coerce")
    sed_ap = (
        (sed_ap_raw == 1).sum()
        if sed_ap_raw.notna().any()
        else pd.NA
    )

    return pd.Series({
        "gt_sedentary_s": sed_gt,
        "ap_total_steps": step_ap,
        "ap_sedentary_s": sed_ap,
    })

summary = merged_df.groupby(["id", "obs"]).apply(
    summarise_group, include_groups=False
).reset_index()

print(summary.to_string())

summary_path = r"C:\Users\HELIOS-300\Desktop\WAVES\AM Full Code\summary_am_testing.csv"
summary.to_csv(summary_path, index=False)
print(f"\nExported summary ({len(summary)} rows) to {summary_path}")


      id    obs  gt_sedentary_s  ap_total_steps ap_sedentary_s
0   AM01    DO1          7175.0           630.0         3159.0
1   AM01    DO2          3041.0            40.0            0.0
2   AM02    DO1           728.0           818.0          814.0
3   AM02    DO2             0.0          5640.0            0.0
4   AM03    DO1          6066.0          1068.0         6088.0
5   AM03    DO2          1620.0          2220.0         1681.0
6   AM04    DO1          1215.0          1006.0         1209.0
7   AM04    DO2          1075.0          6938.0         1070.0
8   AM05    DO2          6179.0           366.0         6176.0
9   AM06    DO1           103.0          2202.0          309.0
10  AM06    DO2             0.0          9086.0            0.0
11  AM07    DO1           388.0           956.0          386.0
12  AM07    DO2           568.0          2350.0          553.0
13  AM08  DO1_a             2.0          1812.0          609.0
14  AM08  DO1_b             0.0          1810.0        